# Implementing Approach

In the following sections, I will perform implement the approach proposed in stage 3, which consists of:

1. Removing "label 2" examples as they are cases of borderline PCL which even human annotators were unable to agree on. Thus, forcing it into any one class is likely to just introduce noise into the dataset. As shown in the EDA, they also only 1.38% of the dataset, and thus can be removed without sacrificing data quantity. This removal and classification of the remaining data into binary 0/1 No-PCL / PCL labels has been done in `data/filtered_pcl_task_data.csv`.

2. Truncating texts which exceed the allowed token size of RoBERTa (512 tokens) to the last sentence <512 tokens. There were 2 such examples of lengths 534 and 934 tokens respectively. 

3. Class weighting: In the model's loss function, there will be greater weightage assigned to the misclassification of PCL examples, forcing the model to optimise for detecting patterns within PCL. As our primary metric is F1-score, this will likely be achieved by performing a weighted-average of the F1 scores of both classes, skewed more toward the PCL side.

4. Performing Layer-wise Learning Rate Decay (LLRD) within the RoBERTa fine-tuning.

In [1]:
!pip install -r requirements.txt

In [2]:
import pandas as pd 
import numpy as np 
import spacy
import matplotlib
import transformers
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import torch
import torch.nn as nn

/home/azureuser/60035_nlp_cw/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import AutoModelForSequenceClassification

MODEL = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=1
)


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 577.70it/s, Materializing param=deberta.encoder.rel_embeddings.weight]                     
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MIS

In [4]:
DATA_FOLDER_PATH = "data" 
DATASET_PATH = f"{DATA_FOLDER_PATH}/filtered_pcl_task_data.csv"

data = pd.read_csv(DATASET_PATH)
data.head()

# We have a Null value, hence we drop it 
null_values = data[["text", "label"]].isnull().sum() > 0
print(null_values)
if null_values.any():
    data = data.dropna(subset=["text", "label"]).reset_index(drop=True)

text      True
label    False
dtype: bool


In [5]:
# Splitting data into train and dev sets
TRAIN_PATH = f"{DATA_FOLDER_PATH}/train_semeval_parids-labels.csv"
DEV_PATH = f"{DATA_FOLDER_PATH}/dev_semeval_parids-labels.csv"
DATASET_FEATURES = ["id", "text", "label"]

train_df = pd.read_csv(TRAIN_PATH)
dev_df = pd.read_csv(DEV_PATH)


train_data = data[data["id"].isin(train_df["par_id"])][DATASET_FEATURES]
val_data = data[data["id"].isin(dev_df["par_id"])][DATASET_FEATURES]

# Validate we have both labels in both sets
assert(train_data["label"].nunique() == 2 and val_data["label"].nunique() == 2)

display(train_data.head())
display(val_data.head())

,id,text,label
0,1,"We 're living in times of absolute insanity , ...",0
1,2,"In Libya today , there are countless number of...",0
2,3,White House press secretary Sean Spicer said t...,0
3,4,Council customers only signs would be displaye...,0
4,5,""" Just like we received migrants fleeing El Sa...",0


,id,text,label
105,107,"His present "" chambers "" may be quite humble ,...",1
148,151,10:41am - Parents of children who died must ge...,1
151,154,When some people feel causing problem for some...,1
154,157,We are alarmed to learn of your recently circu...,1
182,187,""" We share a global responsibility to respond ...",1


okay give me the code to do this fine tuning from start to end. I want:
 - Class weighting (proportional to sqrt of PCL : no-PCL ratio)
 - LLRD
 - RoBERTa-base model


In [6]:
# Determine class imbalance and weights to apply

train_labels = train_data["label"].values
n_neg = np.sum(train_labels == 0)
n_pos = np.sum(train_labels == 1)

print(f"Non-PCL: {n_neg}, PCL: {n_pos}, Ratio: {n_neg/n_pos:.2f}")

# Square root weighting to avoid being too aggressive
pos_class_weight = torch.tensor(np.sqrt(n_neg / n_pos), dtype=torch.float)
print(f"Class weights: {pos_class_weight}")

Non-PCL: 7581, PCL: 668, Ratio: 11.35
Class weights: 3.36879825592041


In [7]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

TRAIN_BATCH_SIZE = 16
NUM_EPOCHS = 20

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
    )

# Pads inputs to the longest sequence in batch. Required to ensure consistent batch sizes
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

train_dataset = Dataset.from_pandas(train_data[["text", "label"]])
val_dataset = Dataset.from_pandas(val_data[["text", "label"]])

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")

# Convert columns to torch tensors
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 2075/2075 [00:00<00:00, 9302.01 examples/s]


In [8]:
# Define custom trainer to perform class-weighted BCE 

from transformers import Trainer

class WeightedBCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float().view(-1)
        outputs = model(**inputs)
        logits = outputs.logits.view(-1)

        pos_weight = pos_class_weight.to(logits.device).view(())  # scalar
        loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)(logits, labels)

        return (loss, outputs) if return_outputs else loss


In [9]:
from transformers import get_cosine_schedule_with_warmup

def get_optimizer_params(model, base_lr, weight_decay=0.01, layerwise_lr_decay=0.9):
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = []
    
    # helper to split params into decay/no-decay
    def get_layer_params(params, lr):
        return [
            {"params": [p for n, p in params if not any(nd in n for nd in no_decay)], 
             "weight_decay": weight_decay, "lr": lr},
            {"params": [p for n, p in params if any(nd in n for nd in no_decay)], 
             "weight_decay": 0.0, "lr": lr}
        ]

    # 1. Head (Classifier & Pooler)
    head_params = [(n, p) for n, p in model.named_parameters() if "classifier" in n or "pooler" in n]
    optimizer_grouped_parameters.extend(get_layer_params(head_params, base_lr))

    # 2. Encoder Layers (11 down to 0)
    current_lr = base_lr
    for i in range(11, -1, -1):
        current_lr *= layerwise_lr_decay
        layer_params = [(n, p) for n, p in model.named_parameters() if f"encoder.layer.{i}." in n]
        optimizer_grouped_parameters.extend(get_layer_params(layer_params, current_lr))

    # 3. Embeddings
    current_lr *= layerwise_lr_decay
    emb_params = [(n, p) for n, p in model.named_parameters() if "embeddings" in n]
    optimizer_grouped_parameters.extend(get_layer_params(emb_params, current_lr))

    return optimizer_grouped_parameters

In [10]:
from torch.optim import AdamW

# Setup the grouped parameters
grouped_params = get_optimizer_params(model, base_lr=2e-5)

# Create the Optimizer
optimizer = AdamW(grouped_params, lr=2e-5, eps=1e-6)

# Calculate real steps for the scheduler
steps_per_epoch = len(train_dataset) // TRAIN_BATCH_SIZE
total_steps = steps_per_epoch * NUM_EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(total_steps * 0.1), 
    num_training_steps=total_steps
)

In [11]:
# Define function to compute F1 score from raw model outputs

from sklearn.metrics import f1_score

def compute_f1_score(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    logits = np.asarray(logits).reshape(-1)
    labels = np.asarray(labels).reshape(-1).astype(int)

    probs = 1 / (1 + np.exp(-logits))

    # search thresholds
    thresholds = np.linspace(0.001, 0.5, 100)
    best_f1, best_t = -1.0, 0.5
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    return {"f1": best_f1, "best_threshold": best_t}


    # logits = eval_pred.predictions
    # labels = eval_pred.label_ids

    # logits = np.asarray(logits).reshape(-1)
    # labels = np.asarray(labels).reshape(-1).astype(int)

    # probs = 1 / (1 + np.exp(-logits))
    # preds = (probs >= 0.5).astype(int)

    # return {"f1": f1_score(labels, preds)}

In [12]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./checkpoints", # Store checkpoints in this directory
    num_train_epochs=20,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    metric_for_best_model="f1", # Evaluate performance using F1 score on validation set
    greater_is_better=True, # Higher F1 score on val set is better
    load_best_model_at_end=True,
    fp16=False,
    bf16=False,
    max_grad_norm=1.0,
)

In [13]:
trainer = WeightedBCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,    # Dynamically pads received inputs
    compute_metrics=compute_f1_score,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    optimizers=(optimizer, scheduler) # <--- This is where the magic happens
)

trainer.train()

Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 19.81 MiB is free. Process 3799 has 12.02 GiB memory in use. Including non-PyTorch memory, this process has 2.52 GiB memory in use. Of the allocated memory 2.35 GiB is allocated by PyTorch, and 28.76 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Plot results after completion of training

log_history = trainer.state.log_history
train_logs = [x for x in log_history if "loss" in x and "eval_loss" not in x]
eval_logs  = [x for x in log_history if "eval_loss" in x]

train_loss = [x["loss"] for x in train_logs]
eval_loss  = [x["eval_loss"] for x in eval_logs]
eval_f1    = [x["eval_f1"] for x in eval_logs]
epochs     = [x["epoch"] for x in eval_logs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_loss, label="Train Loss")
ax1.plot(epochs, eval_loss, label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.set_title("Loss")

ax2.plot(epochs, eval_f1, label="Val F1", color="green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("F1")
ax2.legend()
ax2.set_title("Validation F1")

plt.tight_layout()
plt.show()